In [2]:
import pandas as pd
import joblib

def predict_production(new_data, model_path='production_model.joblib'):
    """
    Loads the saved model and predicts actual_qty for new data.
    new_data: A pandas DataFrame containing the pre-production features.
    """
    # 1. Load the saved pipeline
    pipeline = joblib.load(model_path)
    
    # 2. Apply the exact same feature engineering
    # We use .copy() to avoid modifying the original input dataframe
    df_pred = new_data.copy()
    df_pred['theoretical_max_production'] = (df_pred['working_hours'] * 60) / df_pred['total_duration_per_shoe_mins']
    df_pred['machine_worker_ratio'] = df_pred['machine_count'] / df_pred['no_of_workers']
    df_pred['avg_hours_per_worker'] = df_pred['working_hours'] / df_pred['no_of_workers']
    
    # 3. Predict
    predictions = pipeline.predict(df_pred)
    
    # Return rounded integers since you are predicting a quantity of shoes
    return predictions.round().astype(int)

# --- EXAMPLE USAGE ---
if __name__ == "__main__":
    # Example of a new incoming production order (notice planned_qty is missing!)
    new_order = pd.DataFrame([{
        'financial_month': 'Apr',
        'week_no': 4,
        'week_of_month': 4,
        'factory_unit': 'Factory Unit 01',
        'product_family': 'Ladies',
        'item_name': 'BINOLI (VAST) - BROWN',
        'size_id': 10,
        'grade': 'A',
        'rate': 4500.00,
        'no_of_workers': 18,
        'working_hours': 800.5,
        'total_duration_per_shoe_mins': 120.0,
        'machine_count': 5,
        'order_category': 'Others'
    }])

    # Get the prediction
    predicted_qty = predict_production(new_order)
    
    print(f"Predicted Actual Production Qty: {predicted_qty[0]} shoes")

Predicted Actual Production Qty: 43 shoes
